In [1]:
# pip install openai neo4j numpy
import os
import sys
from typing import List, Dict, Any, Tuple

import numpy as np
from neo4j import GraphDatabase, basic_auth
from openai import OpenAI

# =========================
# Config (uses your env vars)
# =========================
URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DB = os.getenv("NEO4J_DATABASE", "neo4j")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
EMBED_MODEL = "text-embedding-3-small"   # same model as before

# =========================
# Clients
# =========================
driver = GraphDatabase.driver(URI, auth=basic_auth(USER, PASSWORD))
oa_client = OpenAI(api_key=OPENAI_API_KEY)

# =========================
# Embeddings
# =========================
def embed_text(text: str, model: str = EMBED_MODEL) -> List[float]:
    resp = oa_client.embeddings.create(model=model, input=[text])
    return resp.data[0].embedding

# =========================
# Similarity
# =========================
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0.0:
        return 0.0
    return float(np.dot(a, b) / denom)

# =========================
# Neo4j fetch
# =========================
def load_all_documents_with_embeddings(limit: int | None = None) -> List[Dict[str, Any]]:
    """
    Fetch all Document nodes that have an embedding.
    """
    cypher = """
    MATCH (d:Document)
    WHERE d.embedding IS NOT NULL AND size(d.embedding) > 0
    RETURN d.id AS id, d.title AS title, d.text AS text, d.embedding AS embedding
    """
    if limit:
        cypher += " LIMIT $limit"
    with driver.session(database=DB) as session:
        result = session.run(cypher, limit=limit)
        rows = []
        for r in result:
            emb = r["embedding"]
            if isinstance(emb, list) and emb and isinstance(emb[0], (int, float)):
                rows.append({
                    "id": r["id"],
                    "title": r["title"],
                    "text": r["text"],
                    "embedding": emb
                })
        return rows

# =========================
# Main flow
# =========================
def top_k_similar(query_text: str, docs: List[Dict[str, Any]], k: int = 3) -> List[Tuple[Dict[str, Any], float]]:
    q_emb = np.array(embed_text(query_text), dtype=np.float32)
    scored: List[Tuple[Dict[str, Any], float]] = []
    for d in docs:
        emb = np.array(d["embedding"], dtype=np.float32)
        score = cosine_similarity(q_emb, emb)
        scored.append((d, score))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

if __name__ == "__main__":
    user_paragraph = (
         "The popular US television series, The OC, ran quite successfully for several years. Although it was warmly received, it ended after producing and broadcasting a total of 5 seasons")
    if not user_paragraph:
        print("[ERROR] No input paragraph provided.")
        sys.exit(1)

    print("[INFO] Fetching documents with embeddings from Neo4j…")
    docs = load_all_documents_with_embeddings()
    if not docs:
        print("[WARN] No documents with embeddings found in the database.")
        sys.exit(0)

    print(f"[INFO] Loaded {len(docs)} documents.")
    print("[INFO] Embedding query and ranking…")
    top5 = top_k_similar(user_paragraph, docs, k=3)

    print("\nTop 3 matches:")
    for i, (doc, score) in enumerate(top5, 1):
        title = doc.get("title") or "(no title)"
        doc_id = doc.get("id")
        print(f"{i}. id={doc_id} | title={title} | cosine_sim={score:.4f}")


[INFO] Fetching documents with embeddings from Neo4j…


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownLabelWarning} {category: UNRECOGNIZED} {title: The provided label is not in the database.} {description: One of the labels in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing label name is: Document)} {position: line: 2, column: 14, offset: 14} for query: '\n    MATCH (d:Document)\n    WHERE d.embedding IS NOT NULL AND size(d.embedding) > 0\n    RETURN d.id AS id, d.title AS title, d.text AS text, d.embedding AS embedding\n    '


[WARN] No documents with embeddings found in the database.


SystemExit: 0

/home/sbhavsar/anaconda3/envs/poisonedRAG/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [32]:
# =========================
# Neo4j: fetch nodes (doc_ids ∋ any(topIds)) + neighbors
# =========================
def fetch_nodes_and_neighbors_by_docids(doc_ids: list[str]) -> list[dict]:
    """
    Returns one row per (node, neighbor) pair. If a node has no neighbors,
    it still appears once with rel_type=None and neighbor_id=None.
    """
    cypher = """
    // nodes whose doc_ids contain any of the given doc_ids
    MATCH (n)
    WHERE n.doc_ids IS NOT NULL
      AND any(x IN n.doc_ids WHERE x IN $docIds)

    // pull neighbors (optional)
    OPTIONAL MATCH (n)-[r]-(m)
    WITH n, r, m
    RETURN
      coalesce(n.id, elementId(n))  AS node_id,
      labels(n)                     AS node_labels,
      n.doc_ids                     AS node_doc_ids,
      n.titles                      AS node_titles,
      type(r)                       AS rel_type,
      CASE
        WHEN r IS NULL THEN NULL
        WHEN startNode(r) = n THEN 'OUT'
        ELSE 'IN'
      END                           AS direction,
      coalesce(m.id, elementId(m))  AS neighbor_id,
      labels(m)                     AS neighbor_labels
    ORDER BY node_id, rel_type, neighbor_id
    """
    with driver.session(database=DB) as session:
        return session.run(cypher, {"docIds": doc_ids}).data()

def group_neighbors(rows: list[dict]) -> dict:
    """
    Groups rows by node_id into:
      { node_id: {labels, doc_ids, titles, neighbors:[{rel,dir,id,labels}], ... } }
    """
    out = {}
    for r in rows:
        nid = r["node_id"]
        entry = out.setdefault(nid, {
            "labels": r.get("node_labels", []),
            "doc_ids": r.get("node_doc_ids", []),
            "titles":  r.get("node_titles", []),
            "neighbors": []
        })
        if r.get("rel_type") is not None:
            entry["neighbors"].append({
                "rel": r["rel_type"],
                "dir": r["direction"],
                "id":  r["neighbor_id"],
                "labels": r.get("neighbor_labels", [])
            })
    return out


In [33]:
# collect the winning document ids (e.g., ["doc618", "doc589", ...])
top_doc_ids = [doc["id"] for (doc, _score) in top5]

print("\n[INFO] Finding graph nodes whose doc_ids contain any of:", top_doc_ids)
rows = fetch_nodes_and_neighbors_by_docids(top_doc_ids)
grouped = group_neighbors(rows)

list_ids=[]

# pretty print
for i, (nid, info) in enumerate(grouped.items(), 1):
    print(f"\n=== Node {i} ===")
    print(f"node_id: {nid}")
    list_ids.append(nid)
    print(f"labels : {info['labels']}")
    print(f"doc_ids: {info['doc_ids']}")
    print(f"titles : {info['titles']}")
    if not info["neighbors"]:
        print("neighbors: (none)")
    else:
        print("neighbors:")
        for nb in info["neighbors"]:
            if nb["dir"] == "IN":
                print(f"  {nb['id']} -[{nb['rel']}]-> {nid}  ({nb['labels']})")
            else:
                print(f"  {nid} -[{nb['rel']}]-> {nb['id']}  ({nb['labels']})")



[INFO] Finding graph nodes whose doc_ids contain any of: ['doc642', 'doc635', 'doc641']

=== Node 1 ===
node_id: abc
labels : ['organization']
doc_ids: ['doc635']
titles : []
neighbors:
  lost -[AIRED_ON]-> abc  (['entertainment'])

=== Node 2 ===
node_id: campaign_to_save_the_oc
labels : ['event']
doc_ids: ['doc642']
titles : []
neighbors:
  campaign_to_save_the_oc -[HAS_CONTENT]-> val_500000  (['value_number'])
  campaign_to_save_the_oc -[INSTANCE_OF]-> the_oc_season_4  (['work'])

=== Node 3 ===
node_id: cbs
labels : ['organization']
doc_ids: ['doc635']
titles : []
neighbors:
  criminal_minds -[AIRED_ON]-> cbs  (['entertainment'])

=== Node 4 ===
node_id: criminal_minds
labels : ['entertainment']
doc_ids: ['doc635']
titles : []
neighbors:
  criminal_minds -[AIRED_ON]-> cbs  (['organization'])

=== Node 5 ===
node_id: csi
labels : ['entertainment']
doc_ids: ['doc641']
titles : []
neighbors:
  the_oc_season_4 -[COMPETED_WITH]-> csi  (['work'])

=== Node 6 ===
node_id: cw_television_n

In [21]:
list_ids

['center',
 'circles_intersection',
 'dihedral_group_order_6_d3',
 'equilateral_arches',
 'equilateral_triangle',
 'euclidean_geometry',
 'platonic_solids',
 'regular_polygon',
 'regular_tetrahedron',
 'regular_uniform_polyhedra',
 'rotational_symmetry_order_3',
 'triangular_tiling',
 'val_3_lines_reflection',
 'value_text_60_degrees']